# What We Collected & How to Use It

This notebook covers two things:

1. **Projects in this dataset** — what ORACC projects we collected, their catalogues, and how to load them
2. **Reference datasets bundled with the package** — provenance mappings, historical periods, sign list, POS tags, and language codes

| Data | Location | How to access |
|---|---|---|
| **Project catalogues** (raw ORACC metadata, 1 row/text) | `enriched_data/catalogues/` | `load_project_catalogue()` |
| **Word CSVs** (parsed text content, 1 row/word) | `enriched_data/oracc_csvs/` | `load_word_csvs_from_dir()` |
| **Provenance (cities → Pleiades)** | Bundled CSV | `reference_data.get_provenance()` |
| **Historical periods → year ranges** | Bundled CSV | `reference_data.get_period_mapping()` |
| **Cuneiform sign readings** | Bundled CSV | `reference_data.get_sign_list()` |
| **POS tag meanings** | Bundled CSV | `reference_data.get_pos_tags()` |
| **Language codes** | Bundled CSV | `reference_data.get_languages()` |

In [1]:
from oracc_parser import reference_data, load_project_catalogue
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR, CATALOGUE_DIR
import pandas as pd

## 1. Projects in This Dataset

The Zenodo record (doi: 10.5281/zenodo.19834993) contains pre-processed data for 138 ORACC projects — a subset of the full ORACC corpus. For each project we provide:

- A **catalogue CSV** (`enriched_data/catalogues/{project_slug}.csv`) with one row per text and all raw ORACC metadata fields preserved
- A folder of **word CSVs** (`enriched_data/oracc_csvs/{project_slug}/`) with one CSV per text and one row per word

The summary below is built directly from the catalogue files currently on disk (will depend on what you already downloaded locally in the other notebooks).

> For the live count from ORACC's servers, use `reference_data.get_live_project_list()`.

In [2]:
# Summary of all collected projects
rows = []
for cat_file in sorted(CATALOGUE_DIR.glob("*.csv")):
    df = load_project_catalogue(cat_file)
    project = df["project"].iloc[0] if not df.empty else cat_file.stem
    n_texts = len(df)
    csv_dir = WORD_CSV_DIR / cat_file.stem
    rows.append({"project": project, "catalogue_entries": n_texts})

dataset_summary = pd.DataFrame(rows)
print(f"{len(dataset_summary)} projects in this dataset")
print(f"Total catalogue entries: {dataset_summary['catalogue_entries'].sum():,}")
display(dataset_summary)

138 projects in this dataset
Total catalogue entries: 34,102


,project,catalogue_entries
0,adsd/adart1,417
1,adsd/adart2,417
2,adsd/adart3,417
3,adsd/adart5,106
4,adsd/adart6,180
...,...,...
133,tcma/tsa1,17
134,tcma/tsh1,262
135,tcma/ugarit,5
136,unknown,0


In [3]:
# Search collected projects by keyword
SEARCH = "saa"
matches = dataset_summary[dataset_summary["project"].str.contains(SEARCH, case=False, na=False)]
print(f"{len(matches)} projects matching '{SEARCH}':")
display(matches)

22 projects matching 'saa':


,project,catalogue_entries
84,saao/saa01,265
85,saao/saa02,15
86,saao/saa03,52
87,saao/saa04,354
88,saao/saa05,300
89,saao/saa06,350
90,saao/saa07,219
91,saao/saa08,567
92,saao/saa09,11
93,saao/saa10,389


In [8]:
# Load and inspect a specific project's catalogue
PROJECT = "saao/saa01"
slug = PROJECT.replace("/", "-")

catalogue = load_project_catalogue(CATALOGUE_DIR / f"{slug}.csv")
print(f"Catalogue for {PROJECT}: {len(catalogue)} texts")
print(f"Fields: {list(catalogue.columns)}")
display(catalogue.head())

Catalogue for saao/saa01: 265 texts
Fields: ['langs', 'project', 'designation', 'atae_lists', 'has_date', 'ancient_author', 'ancient_recipient', 'archive', 'cdli_excavation_no', 'cdli_id', 'cdli_museum_no', 'ch_name', 'ch_no', 'ch_num_name', 'collection', 'comp_wit', 'credits', 'date_of_origin', 'dialect', 'display_name', 'dossier_list', 'edited_twice', 'editorial_comments', 'excavation_no', 'genre', 'id_text', 'language', 'material', 'museum_no', 'nl_no', 'object_type', 'oracc_header', 'period', 'please_cite', 'pleiades_id', 'primary_publication', 'provenience', 'pleiades_coord', 'publication_history', 'regnal_dates', 'rework', 'ruler', 'saa_attribution', 'saa_cdli_excavation', 'saa_cdli_id', 'saa_cdli_museum', 'saao_lists', 'script', 'script_remarks', 'script_type', 'secondary_record', 'sender_loc', 'short_title', 'title', 'vol_title', 'volume', 'trans', 'text_id', 'date', 'proposed_date', 'dossier', 'other_pub', 'gpa_no', 'accession_no', 'cdli_accession_no', 'ct_53_no', 'pleiades_se

,langs,project,designation,atae_lists,has_date,ancient_author,ancient_recipient,archive,cdli_excavation_no,cdli_id,...,gpa_no,accession_no,cdli_accession_no,ct_53_no,pleiades_sender_loc,saa_cdli_accession,sender_loc_coordinates,public,images,abl_no
0,0x08000000,saao/saa01,SAA 01 175,atae/saao:P224395,no,Adda-hati,the king,"006 - Northwest Palace, Room ZT 4",ND 02381,P224395,...,,,,,,,,,,
1,0x08000000,saao/saa01,SAA 01 026,atae/saao:P224403,no,the king,go[vernor] (of Calah),"006 - Northwest Palace, Room ZT 4",ND 02408,P224403,...,,,,,,,,,,
2,0x08000000,saao/saa01,SAA 01 176,atae/saao:P224417,no,Adda-hati,the king,"006 - Northwest Palace, Room ZT 4",ND 02437,P224417,...,,,,,,,,,,
3,0x08000000,saao/saa01,SAA 01 172,atae/saao:P224431,no,[Bel-duri],[the king],"006 - Northwest Palace, Room ZT 4",ND 02495,P224431,...,,,,,,,,,,
4,0x08000000,saao/saa01,SAA 01 032,atae/saao:P224433,no,S[ennacherib],the king,"006 - Northwest Palace, Room ZT 4",ND 02608,P224433,...,,,,,,,,,,


> ⚠️ Note: catalogue entries across ORACC projects are not consistent. While some columns appear in all projects (e.g. `langs`, `project`, etc.), others do not.

## 2. ORACC Project Catalogue (Bundled Metadata)

A static snapshot of metadata for **all known ORACC projects** — names, languages, URLs, and whether a project has parseable text content. This is bundled with the package and loads without any download.

In [4]:
projects = reference_data.get_projects_metadata()
print(f"🌍 {len(projects)} ORACC projects in the bundled catalogue")
display(projects.head(10))

🌍 221 ORACC projects in the bundled catalogue


,Project,Link,Project_Name,Languages,Is_Umbrella_Project,Is_Text_Folder_Empty,Last_Check
0,adsd: adsd/Astronomical Diaries and Related Texts,http://oracc.museum.upenn.edu/adsd,adsd,Akkadian,yes,yes,14/05/2025
1,adsd/adart1: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart1,adsd/adart1,Akkadian,no,no,14/05/2025
2,adsd/adart2: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart2,adsd/adart2,Akkadian,no,no,14/05/2025
3,adsd/adart3: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart3,adsd/adart3,Akkadian,no,no,14/05/2025
4,adsd/adart5: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart5,adsd/adart5,Akkadian,no,no,14/05/2025
5,adsd/adart6: adsd/Astronomical Diaries and Rel...,http://oracc.museum.upenn.edu/adsd/adart6,adsd/adart6,Akkadian,no,no,14/05/2025
6,AEMW: Akkadian in the Eastern Mediterranean World,https://oracc.museum.upenn.edu/aemw,aemw,"Akkadian, Ugaritic",yes,no,14/05/2025
7,aemw/idrimi: Statue of Idrimi,http://oracc.museum.upenn.edu/aemw/alalakh/idrimi,aemw/alalakh/idrimi,Akkadian,no,no,14/05/2025
8,aemw/amarna: The Amarna Letters,http://oracc.museum.upenn.edu/aemw/amarna,aemw/amarna,Akkadian,no,no,14/05/2025
9,aemw/ugarit — AEMW Ugarit Corpus,https://oracc.museum.upenn.edu/aemw/ugarit/corpus,aemw/ugarit,"Akkadian, Ugaritic",no,no,14/05/2025


### 2.1 Provenance — Tablet Find Spots

ORACC tablets come with a raw `provenience` string (e.g. `"Nineveh"`, `"Assur"`).
We pre-built a mapping from these raw strings to:
- A **normalized city name**
- A **[Pleiades](https://pleiades.stoa.org/) ID** and coordinates (lat/lon)

This was built by:
1. Collecting all unique provenience strings from ORACC
2. Matching them to Pleiades gazeteer entries via API
3. Hand-verifying ambiguous cases

By default `get_provenance()` returns **only rows with a confirmed Pleiades ID**.
This is also what the pipeline uses to attach coordinates to parsed tablets.

In [5]:
# By default: only rows with a confirmed numeric Pleiades ID
prov = reference_data.get_provenance()
print(f"✅ {len(prov)} provenance mappings with confirmed Pleiades IDs")
display(prov.head(20))

✅ 262 provenance mappings with confirmed Pleiades IDs


,raw_provenience,normalized_city,pleiades_id,pleiades_title,lat,lon
0,Acharneh (Tunip),Tunip,668200,Asharne,35.2833333,36.4
1,Adab,Adab,787747618,Adab,31.9509002901,45.62333875145
2,Akhetaten (Mod. Amarna),el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
3,Akhetaten (mod. el-Amarna),el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
4,Akko,Akko,678010,Ake/Ptolemais,32.92128675,35.079835
5,Alalakh,Alalakh,309866128,Alalaḫ,36.23807544872566,36.38383192027712
6,Amarna,el-Amarna,149576487,Amarna,27.644099597341675,30.89818203754354
7,Ana,Ana,893936,Anatho,34.4678,41.9794
8,Antakya,Antakya,658381,Antiochia/Theoupolis,36.2225515,36.183214
9,Aqar Quf (Dur-Kurigalzu),Dur-Kurigalzu,893988,Dur-(Kuri)galzu,33.35607304048341,44.197416520972745


In [6]:
# To see ALL mappings (including uncertain/unmatched):
from oracc_parser.utils.paths import get_provenience
prov_all = get_provenience(pleiades_only=False)
print(f"🗺️  Total provenance mappings: {len(prov_all)}")
print(f"   With Pleiades ID:           {len(prov)} ({len(prov)/len(prov_all):.0%})")
print(f"   Without Pleiades ID:        {len(prov_all) - len(prov)} (uncertain / unidentified)")

🗺️  Total provenance mappings: 333
   With Pleiades ID:           262 (79%)
   Without Pleiades ID:        71 (uncertain / unidentified)


In [7]:
# Cities with confirmed Pleiades IDs
cities = prov['normalized_city'].dropna().unique()
print(f"{len(cities)} unique cities with Pleiades coordinates:")
for c in sorted(cities)[:30]:
    print(f"  {c}")

157 unique cities with Pleiades coordinates:
  Adab
  Akko
  Alalakh
  Ana
  Antakya
  Arbela
  Arslan Tash
  Arzawa
  Ashdod
  Ashkelon
  Assur
  Babylon
  Bad-Tibira
  Balāwāt
  Bavian
  Beirut
  Beisan
  Ben Shemen
  Bisutun
  Borsippa
  Bost
  Brissa
  Burmarina
  Byblos
  Carchemish
  Cutha
  Daskyleion
  Dawali
  Der
  Dilbat


### 2.2 Historical Periods → Year Ranges

ORACC text metadata includes period names like `"Neo-Assyrian"`, `"Old Babylonian"`, etc.
We pre-saved a mapping from these period names to approximate year ranges.
The pipeline uses this to add `start_year` / `end_year` to every parsed tablet.

In [8]:
periods = reference_data.get_period_mapping()
print(f"{len(periods)} historical periods")
display(periods)

21 historical periods


,period_name,start_year,end_year
0,hellenistic,-323,-30
1,achaemenid,-550,-330
2,Neo-Babylonian,-626,-539
3,Neo-Assyrian,-911,-612
4,Parthian,-247,224
5,Middle Babylonian,-1595,-1155
6,Middle Assyrian,-1363,-912
7,Old Babylonian,-1894,-1595
8,Old Akkadian,-2334,-2154
9,Early Old Babylonian,-1894,-1595


### 2.3 Cuneiform Sign List (8,900+ readings)

This table maps cuneiform sign names to their Unicode code points.
The parser uses it to render transliterated text as actual cuneiform characters (𒀸, 𒈾, etc.).
It was compiled from the ORACC sign list.

In [9]:
signs = reference_data.get_sign_list()
print(f"{len(signs)} sign readings")
display(signs.head(20))

8902 sign readings


,reading,cuneiform
0,ʾu₄,𒀀
1,a,𒀀
2,aia₂,𒀀
3,aya₂,𒀀
4,barₓ,𒌋
5,buniŋₓ,𒆹
6,burₓ,𒋲
7,dur₅,𒀀
8,duru₅,𒀀
9,e₄,𒀀


In [10]:
# Search for a sign reading
SIGN_QUERY = "lugal"  # Try: "an", "dingir", "sar", "gal", "ki"

mask = signs.apply(
    lambda r: SIGN_QUERY.lower() in ' '.join(r.fillna('').astype(str)).lower(),
    axis=1,
)
matches = signs[mask]
print(f"Found {len(matches)} readings for '{SIGN_QUERY}':")
display(matches)

Found 2 readings for 'lugal':


,reading,cuneiform
4885,lugal,𒈗
4886,lugala,𒈗


### 2.4 POS Tags & Language Codes

**POS tags** — ORACC uses a custom tagset (e.g. `PN` = Personal Name, `GN` = Geographical Name).
We presaved a table linking each tag to its human-readable meaning.

In [11]:
pos = reference_data.get_pos_tags()
print(f"There are {len(pos)} POS tags in total. Here are the major ones used for masking:")
major_masks = pos[pos['To mask'] == 'yes'][['POS-tag', 'Meaning', 'Mask as']].drop_duplicates()
for _, row in major_masks.head(10).iterrows():
    print(f"  '{row['POS-tag']}' ({row['Meaning']}) -> masked as {row['Mask as']}")
print("  ...")

There are 56 POS tags in total. Here are the major ones used for masking:
  'MN' (Month Name) -> masked as MN
  'n' (Number) -> masked as NUM
  'CN' (Constellation Name) -> masked as CN
  'PN' (Personal Name) -> masked as PN
  'RN' (Royal Name) -> masked as RN
  'GN' (Geographical Name) -> masked as GN
  'SN' (State Name (city name)) -> masked as GN
  'DN' (Divine Name) -> masked as DN
  'TN' (Temple Name) -> masked as GN
  'NU' (Number) -> masked as NUM
  ...


**Language codes** — ORACC texts are tagged with BCP-47-style language codes
(e.g. `akk-x-neoass` = Neo-Assyrian Akkadian). This table documents what each code means.

In [12]:
langs = reference_data.get_languages()
print(f"There are {len(langs)} language variants. The base languages are:")
base_langs = langs['language_name'].dropna().unique()
for l in sorted(base_langs):
    print(f"  - {l}")

There are 49 language variants. The base languages are:
  - Akkadian
  - Aramaic
  - Canaanite
  - Egyptian
  - Elamite
  - Greek
  - Hittite
  - Hurrian
  - Numbers
  - Persian
  - Proto-Elamite
  - Proto-cuneiform
  - Sumerian
  - Ugaritic
  - Unclear
  - Urartian


## What's next?

- **Quickstart:** See `01_quickstart.ipynb` to download data, parse a project from CSVs, and export results
- **Configure parsing:** See notebook `03_configure_and_export.ipynb` for masking PNs, dropping broken signs, and changing output formats.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.